# Demo: orquestador multi-agente con supervisor

Este notebook muestra el **flujo de delegación** del orquestador del Módulo 6:
`supervisor → investigador → supervisor → analista → … → sintetizador`.

- **Parte 1 — con LLM real** (Anthropic): una consulta que necesita dos dominios (investigar datos en la
  documentación y calcular con ellos). Se ve cómo decide el supervisor, quién aporta qué y cómo se cierra.
- **Parte 2 — sin LLM**: las reglas de validación del supervisor actuando sobre el grafo real (cierre
  prematuro, supervisor obstinado, tope de pasos).

Requiere `ANTHROPIC_API_KEY` en el `.env` (solo la Parte 1). Se ejecuta desde la carpeta del módulo.

In [1]:
import asyncio, logging, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()
logging.basicConfig(level=logging.WARNING)

from graph import build_graph, mermaid, RECURSION_LIMIT
from llm import get_chat_model
from main import CONSULTA_DEMO
from runner import ejecutar

## Topología

Un **supervisor** central (router inteligente) decide a quién delegar; los especialistas siempre le devuelven
el control. La arista condicional del supervisor devuelve un `Literal` con los nombres de nodo válidos.

In [2]:
graph = build_graph(get_chat_model("anthropic", max_tokens=2048))
g = graph.get_graph()
print("Nodos:", [n for n in g.nodes if not n.startswith("__")])
print("Aristas desde el supervisor (condicionales):", sorted(e.target for e in g.edges if e.source == "supervisor"))
print(f"Límite de recursión del grafo: {RECURSION_LIMIT}")
display(Markdown("```mermaid\n" + mermaid(graph) + "\n```"))

Nodos: ['supervisor', 'investigador', 'analista', 'sintetizador']
Aristas desde el supervisor (condicionales): ['analista', 'investigador', 'sintetizador']
Límite de recursión del grafo: 25


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	investigador(investigador)
	analista(analista)
	sintetizador(sintetizador)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analista --> supervisor;
	investigador --> supervisor;
	supervisor -.-> analista;
	supervisor -.-> investigador;
	supervisor -.-> sintetizador;
	sintetizador --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## Parte 1 — Delegación con LLM real

In [3]:
display(Markdown("**Consulta:** " + CONSULTA_DEMO))

**Consulta:** Según nuestra documentación de respuesta a incidentes, ¿cuál es el tiempo de respuesta de cada nivel de severidad (SEV1, SEV2 y SEV3)? Con esos datos calculá: (a) cuántas actualizaciones de estado debería publicar el equipo durante un incidente SEV1 que dura lo mismo que el RTO de las bases de datos de producción, y (b) el tiempo de respuesta total acumulado si en un mes ocurren 1 SEV1, 2 SEV2 y 3 SEV3.

In [4]:
estado, traza = await ejecutar(graph, CONSULTA_DEMO)
print("FLUJO:", " -> ".join(traza.flujo()))
print(f"Duración: {traza.duracion_s:.0f} s · pasos del supervisor: {estado['pasos']} · tarea completa: {estado['task_completed']}")

FLUJO: supervisor -> investigador -> supervisor -> analista -> supervisor -> sintetizador
Duración: 30 s · pasos del supervisor: 3 · tarea completa: True


### Cada decisión del supervisor, con su razón

In [5]:
for d in estado["decisiones"]:
    regla = f" · regla {d['regla_aplicada']}" if d["regla_aplicada"] else ""
    print(f"[paso {d['paso']}] -> {d['siguiente']}{regla}")
    print("   razón:", " ".join(d["razon"].split())[:260])
    if d["instruccion"]:
        print("   instrucción:", " ".join(d["instruccion"].split())[:260])

[paso 1] -> investigador
   razón: La solicitud requiere datos sobre tiempos de respuesta por nivel de severidad (SEV1, SEV2, SEV3) y el RTO de bases de datos de producción, que deben buscarse en la documentación interna. No hay datos previos, por lo que el investigador debe recopilar TODOS los
   instrucción: Buscá en la documentación de respuesta a incidentes: 1. Tiempo de respuesta (en horas o minutos) para incidentes SEV1 2. Tiempo de respuesta (en horas o minutos) para incidentes SEV2 3. Tiempo de respuesta (en horas o minutos) para incidentes SEV3 4. RTO (Obje
[paso 2] -> analista
   razón: El investigador cumplió la rúbrica: (1) COBERTURA: reportó tiempos de respuesta para SEV1/2/3 y el RTO de BD de producción necesario para (a); también la frecuencia de actualización de estado para SEV1 requerida en (a). (2) FUNDAMENTO: cada dato cita la fuente
   instrucción: (a) Calculá cuántas actualizaciones de estado debería publicar el equipo durante un incidente SEV1 que dura 2 horas (RT

### Procedencia: qué agente aportó qué (estado compartido)

In [6]:
for c in estado["contribuciones"]:
    print(f"{c['agente']:<13} v{c['version']} (decisión {c['paso']})  herramientas={c['herramientas_usadas']}")
    if c["fuentes"]:
        print("              fuentes:", c["fuentes"])

investigador  v1 (decisión 1)  herramientas=['listar_documentos', 'buscar_documentacion', 'buscar_documentacion', 'buscar_documentacion']
              fuentes: ['02_respuesta_a_incidentes.md#2. Niveles de severidad', '01_politica_de_backups.md#4. Objetivos de recuperación']
analista      v1 (decisión 2)  herramientas=['convertir_tiempo', 'calcular', 'convertir_tiempo', 'calcular']


### Respuesta final (sintetizador)

In [7]:
display(Markdown(estado["respuesta_final"]))
if estado["advertencias"]:
    print("Advertencias:", estado["advertencias"])

## RESPUESTA SINTETIZADA

### Tiempos de respuesta por nivel de severidad
Según la documentación (02_respuesta_a_incidentes.md#2. Niveles de severidad):
- **SEV1:** 5 minutos
- **SEV2:** 15 minutos
- **SEV3:** 4 horas

---

### (a) Actualizaciones de estado durante SEV1 de 2 horas

El investigador proporcionó que los incidentes SEV1 requieren actualizaciones cada 15 minutos (02_respuesta_a_incidentes.md#2. Niveles de severidad), y que el RTO de bases de datos de producción es de 2 horas (01_politica_de_backups.md#4. Objetivos de recuperación).

El analista calculó: **120 minutos ÷ 15 minutos = 8 actualizaciones**

**Resultado:** El equipo debe publicar **8 actualizaciones de estado** durante el incidente SEV1.

---

### (b) Tiempo de respuesta total acumulado en un mes

Con 1 SEV1, 2 SEV2 y 3 SEV3, el analista calculó:
- 1 × 5 min = 5 min
- 2 × 15 min = 30 min
- 3 × 4 horas (240 min) = 720 min

**Total: 755 minutos = 12 horas y 35 minutos**

---

**Nota:** El investigador también identificó que los incidentes SEV1 deben comunicarse a la página de estado pública en máximo 10 minutos desde su declaración (02_respuesta_a_incidentes.md#2. Niveles de severidad).

## Parte 2 — Reglas de validación del supervisor (sin LLM)

Aquí el supervisor y los especialistas son piezas con guion, pero el **grafo, el estado y las reglas son los
reales**. Sirve para ver cómo se evitan los cierres prematuros y los bucles infinitos.

In [8]:
from supervisor import DecisionLLM, MAX_PASOS

def supervisor_con_guion(*decisiones):
    i = {"n": 0}
    async def decidir(_estado):
        sig, instr = decisiones[min(i["n"], len(decisiones) - 1)]
        i["n"] += 1
        return DecisionLLM(razon="(supervisor con guion)", siguiente=sig, instruccion=instr)
    return decidir

async def investigador(entrada):
    return {"contenido": "SEV1: 5 min; actualización cada 15 min; RTO 2 h.", "herramientas_usadas": ["buscar_documentacion"],
            "fuentes": ["02_respuesta_a_incidentes.md#2. Niveles de severidad"]}

async def analista(entrada):
    return {"contenido": "120 / 15 = 8 actualizaciones.", "herramientas_usadas": ["calcular"], "fuentes": []}

async def sintetizar(estado):
    return "Respuesta final (síntesis de prueba)."

def grafo_de_prueba(*decisiones):
    return build_graph(decidir=supervisor_con_guion(*decisiones), investigador=investigador, analista=analista, sintetizar=sintetizar)

async def mostrar(titulo, *decisiones):
    est, tr = await ejecutar(grafo_de_prueba(*decisiones), "¿Cuántas actualizaciones en un SEV1 de 2 h?")
    print(f"--- {titulo}")
    print("flujo:", " -> ".join(tr.flujo()))
    for d in est["decisiones"]:
        print(f"   paso {d['paso']}: el supervisor pidió '{d['razon']}' -> ruta final: {d['siguiente']}  regla: {d['regla_aplicada']}")
    print()

In [9]:
# 1) El supervisor quiere cerrar de inmediato, sin haber investigado ni calculado nada.
await mostrar("Cierre prematuro", ("sintetizador", ""), ("analista", "calculá"), ("sintetizador", ""))

--- Cierre prematuro
flujo: supervisor -> investigador -> supervisor -> analista -> supervisor -> sintetizador
   paso 1: el supervisor pidió '(supervisor con guion)' -> ruta final: investigador  regla: R4_cierre_sin_investigacion
   paso 2: el supervisor pidió '(supervisor con guion)' -> ruta final: analista  regla: None
   paso 3: el supervisor pidió '(supervisor con guion)' -> ruta final: sintetizador  regla: None



In [10]:
# 2) El supervisor pide al analista antes de tener datos.
await mostrar("Analista sin datos", ("analista", "calculá"), ("sintetizador", ""))

--- Analista sin datos
flujo: supervisor -> investigador -> supervisor -> analista -> supervisor -> sintetizador
   paso 1: el supervisor pidió '(supervisor con guion)' -> ruta final: investigador  regla: R2_analista_requiere_datos
   paso 2: el supervisor pidió '(supervisor con guion)' -> ruta final: analista  regla: R4_cierre_sin_analisis
   paso 3: el supervisor pidió '(supervisor con guion)' -> ruta final: sintetizador  regla: None



In [11]:
# 3) Supervisor obstinado: pide al investigador una y otra vez. Debe cortarse solo.
await mostrar("Supervisor obstinado", ("investigador", "buscá otra vez"))

--- Supervisor obstinado
flujo: supervisor -> investigador -> supervisor -> investigador -> supervisor -> sintetizador
   paso 1: el supervisor pidió '(supervisor con guion)' -> ruta final: investigador  regla: None
   paso 2: el supervisor pidió '(supervisor con guion)' -> ruta final: investigador  regla: None
   paso 3: el supervisor pidió '(supervisor con guion)' -> ruta final: sintetizador  regla: R3_limite_de_refinamientos



## Conclusión

- El supervisor **decide** con una rúbrica (cobertura, fundamento, cómputo verificado, consistencia, entradas
  completas) y **delega** al especialista adecuado; si un aporte no cumple, lo manda a refinar.
- El estado compartido guarda **quién aportó qué** (y con qué herramientas y fuentes), sin perder contexto.
- Las reglas en código impiden cierres prematuros y acotan los refinamientos: el flujo **siempre termina**
  (máx. 6 decisiones y un límite de recursión).